In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
import numpy as np

from testdata import mvn_with_correlation
x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)

In [7]:
import numpy as np
from numba import njit
from numba.experimental import jitclass
from numba.types import int64, float64
from optikon import Propositionalization

### Utility ###
###############

@njit
def argsort_columns(x):
    n, p = x.shape
    out = np.empty((n, p), dtype=np.int64)
    for j in range(p):
        out[:, j] = np.argsort(x[:, j])
    return out

@njit
def max_weighted_support_greedy(x, y, max_depth=5):
    n, p = x.shape
    orders = argsort_columns(x)
    support = np.ones(n, dtype=np.bool)
    support_count = n

    cum_support_count = 0
    non_separable = 0

    v = np.zeros(max_depth, dtype=np.int64)
    s = np.zeros(max_depth, dtype=np.int64)
    t = np.zeros(max_depth, dtype=np.float64)

    current = np.zeros(p, dtype=np.int64) # cursor buffer for order updates
    
    best_sum = np.sum(y)
    num_cond = 0

    for k in range(1, max_depth+1):
        cum_support_count += support_count
        sum_y = np.sum(y[orders[:support_count, 0]])
        best_j, best_i, best_s = -1, -1, 1
        improvement = False
        for j in range(p):
            sum_left, sum_right = 0, sum_y
            for i in range(support_count - 1): 
                # test splits between x^j_i (last left) and x^j_i+1 (first right)
                y_i = y[orders[i, j]]
                sum_left += y_i
                sum_right -= y_i
                if x[orders[i, j], j]==x[orders[i+1, j], j]:
                    non_separable += 1
                    continue

                if sum_left > best_sum:
                    best_i = i
                    best_j = j
                    best_s = -1
                    best_sum = sum_left
                    improvement = True
                elif sum_right > best_sum:
                    best_i = i
                    best_j = j
                    best_s = 1
                    best_sum = sum_right
                    improvement = True

        if not improvement:
            break

        v[k-1] = best_j
        s[k-1] = best_s
        t[k-1] = best_s*(x[orders[best_i, best_j], best_j] + x[orders[best_i + 1, best_j], best_j]) / 2
        num_cond = k

        if best_s == 1: # lower bound
            support[orders[:best_i+1, best_j]] = False
        else: # upper bound
            support[orders[best_i+1:, best_j]] = False

        current[:] = 0
        for i in range(support_count): # need old support count here
            for j in range(p): # can this loop be vectorised?
                if support[orders[i, j]]:
                    orders[current[j], j] = orders[i, j]
                    current[j] += 1

        if best_s == 1: # lower bound
            support_count = support_count - best_i - 1
        else: # upper bound
            support_count = best_i + 1

    res = Propositionalization(v[:num_cond], t[:num_cond], s[:num_cond])
    return res, best_sum, {'cum_support_count': cum_support_count,
                           'non_separable': non_separable}

res, val, stats = max_weighted_support_greedy(x, y)
res.as_conj_str(), val

('x4 <= 0.494 & x1 <= 1.514 & x2 >= -0.140', 21.443390305350473)

In [13]:
@jitclass
class IncrementalWeightedSupport:

    y: float64[:]
    sum_y: float64
    left_value: float64
    right_value: float64

    def __init__(self, y):
        self.y = y
        self.sum_y = np.sum(y)
        self.left_value = 0
        self.right_value = self.sum_y

    def reset_support(self, support):
        self.sum_y = np.sum(y[support])

    def reset_data(self):
        self.left_value = 0
        self.right_value = self.sum_y

    def move_left(self, i):
        yi = self.y[i]
        self.left_value += yi
        self.right_value -= yi

ws = IncrementalWeightedSupport(y)
ws.move_left(0)
ws.move_left(1)
ws.left_value

-0.006374642197908592

In [19]:
@njit
def greedy_maximization(x, obj, max_depth=5):
    n, p = x.shape
    orders = argsort_columns(x)
    support = np.ones(n, dtype=np.bool)
    support_count = n

    cum_support_count = 0
    non_separable = 0

    v = np.zeros(max_depth, dtype=np.int64)
    s = np.zeros(max_depth, dtype=np.int64)
    t = np.zeros(max_depth, dtype=np.float64)

    current = np.zeros(p, dtype=np.int64) # cursor buffer for order updates
    
    best_value = obj.right_value
    num_cond = 0

    for k in range(1, max_depth+1):
        cum_support_count += support_count

        obj.reset_support(orders[:support_count, 0])
        
        best_j, best_i, best_s = -1, -1, 1
        improvement = False
        for j in range(p):

            obj.reset_data()
            
            for i in range(support_count - 1): 
                # test splits between x^j_i (last left) and x^j_i+1 (first right)
                
                obj.move_left(orders[i, j])
                
                if x[orders[i, j], j]==x[orders[i+1, j], j]:
                    non_separable += 1
                    continue

                # if obj.left_value > best_value
                if obj.left_value > best_value:
                    best_i = i
                    best_j = j
                    best_s = -1
                    best_value = obj.left_value
                    improvement = True
                # elif obj.right_value > best_value
                elif obj.right_value > best_value:
                    best_i = i
                    best_j = j
                    best_s = 1
                    best_value = obj.right_value
                    improvement = True

        if not improvement:
            break

        v[k-1] = best_j
        s[k-1] = best_s
        t[k-1] = best_s*(x[orders[best_i, best_j], best_j] + x[orders[best_i + 1, best_j], best_j]) / 2
        num_cond = k

        if best_s == 1: # lower bound
            support[orders[:best_i+1, best_j]] = False
        else: # upper bound
            support[orders[best_i+1:, best_j]] = False

        current[:] = 0
        for i in range(support_count): # need old support count here
            for j in range(p): # can this loop be vectorised?
                if support[orders[i, j]]:
                    orders[current[j], j] = orders[i, j]
                    current[j] += 1

        if best_s == 1: # lower bound
            support_count = support_count - best_i - 1
        else: # upper bound
            support_count = best_i + 1

    res = Propositionalization(v[:num_cond], t[:num_cond], s[:num_cond])
    return res, best_value, {'cum_support_count': cum_support_count,
                           'non_separable': non_separable}

res, val, stats = greedy_maximization(x, IncrementalWeightedSupport(y))
res.as_conj_str(), val

('x4 <= 0.494 & x1 <= 1.514 & x2 >= -0.140', 21.443390305350473)

In [22]:
@jitclass
class IncrementalRelativeWeightedSupport:

    y: float64[:]
    sum_y: float64
    support_count: int64
    left_sum: float64
    right_sum: float64
    left_count: int64
    right_count: int64
    left_value: float64
    right_value: float64

    def __init__(self, y):
        self.y = y
        self.sum_y = np.sum(y)
        self.support_count = len(y)
        self.reset_data()

    def reset_support(self, support):
        self.sum_y = np.sum(y[support])
        self.support_count = len(support)

    def reset_data(self):
        self.left_sum = 0
        self.right_sum = self.sum_y
        self.left_count = 0
        self.right_count = self.support_count
        self.left_value = 0
        self.right_value = self.sum_y / self.support_count

    def move_left(self, i):
        yi = self.y[i]
        self.left_sum += yi
        self.right_sum -= yi
        self.left_count += 1
        self.right_count += 1
        self.left_value = self.left_sum / self.left_count
        self.right_value = self.right_sum / self.right_count

rws = IncrementalRelativeWeightedSupport(y)
rws.move_left(0)
rws.move_left(1)
rws.move_left(2)
rws.move_left(3)
rws.left_value

0.1847370313496033

In [21]:
res, val, stats = greedy_maximization(x, IncrementalRelativeWeightedSupport(y))
res.as_conj_str(), val

('x3 <= -1.598 & x2 <= 0.628', 1.3664634705496859)